# Notebook 1: Build the daily table, in correct local time

Findings so far: the Apple Health export has no location data, and its timestamps are written in one time zone (SF) even for steps taken in Almaty. So:

1. Convert every timestamp to a true instant (UTC).
2. For each record, work out which city it belongs to from my own date lists (checked in that city's local time).
3. Convert to that city's local time, so 9am means 9am where I actually was.
4. Remove duplicate sources, aggregate to hours and days, save `data/daily_hourly.csv`.

Days outside the SF and Almaty lists, and 2 days at the edge of each period, are excluded.

In [12]:
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

EXPORT_PATH = 'data/export.xml'
OUT_PATH = 'data/daily_hourly.csv'
STEP_TYPE = 'HKQuantityTypeIdentifierStepCount'
MIN_STEPS_PER_DAY = 500
BUFFER_DAYS = 2

ALMATY_PERIODS = [
    ('2022-06-13', '2023-08-25'),
    ('2024-04-22', '2024-09-02'),
    ('2025-04-30', '2025-06-03'),
    ('2025-08-26', '2025-12-21'),
]
SF_PERIODS = [
    ('2023-08-26', '2024-04-21'),
    ('2025-06-04', '2025-08-25'),
    ('2025-12-22', '2026-04-25'),
    ('2026-09-02', '2026-10-14'),
]
ALM_NAMED = [(f'ALM{i+1}', a, b) for i, (a, b) in enumerate(ALMATY_PERIODS)]
SF_NAMED = [(f'SF{i+1}', a, b) for i, (a, b) in enumerate(SF_PERIODS)]

In [ ]:
def load_step_records(path):
    """Stream the XML; one row per step record with its full timestamp string"""
    rows = []
    for _, el in ET.iterparse(path, events=('end',)):
        if el.tag == 'Record' and el.get('type') == STEP_TYPE:
            rows.append((el.get('startDate'), float(el.get('value')), el.get('sourceName')))
        el.clear()
    return pd.DataFrame(rows, columns=['start', 'steps', 'source'])

raw = load_step_records(EXPORT_PATH)
print(f'{len(raw):,} step records')

49,605 step records


In [ ]:
def period_names(dates, named, trim):
    """Name of the period each date falls in (trimmed at both ends), else None"""
    out = pd.Series(None, index=dates.index, dtype=object)
    for name, a, b in named:
        lo = pd.Timestamp(a) + pd.Timedelta(days=trim)
        hi = pd.Timestamp(b) - pd.Timedelta(days=trim)
        out[(dates >= lo) & (dates <= hi)] = name
    return out

def localize(raw):
    """Attach city, period, local time, date and hour to every record that belongs to a city"""
    ts = pd.to_datetime(raw['start'], format='%Y-%m-%d %H:%M:%S %z', utc=True)
    local_sf = ts.dt.tz_convert('America/Los_Angeles').dt.tz_localize(None)
    local_alm = ts.dt.tz_convert('Asia/Almaty').dt.tz_localize(None)

    p_alm = period_names(local_alm.dt.normalize(), ALM_NAMED, BUFFER_DAYS)
    p_sf = period_names(local_sf.dt.normalize(), SF_NAMED, BUFFER_DAYS)
    is_alm = p_alm.notna() & p_sf.isna()
    is_sf = p_sf.notna() & p_alm.isna()
    keep = is_alm | is_sf

    df = raw[keep].copy()
    df['city'] = np.where(is_alm[keep], 'Almaty', 'SF')
    df['period'] = np.where(is_alm[keep], p_alm[keep], p_sf[keep])
    df['local'] = local_alm[keep].where(is_alm[keep], local_sf[keep])
    df['date'] = df['local'].dt.normalize()
    df['hour'] = df['local'].dt.hour
    return df

labeled = localize(raw)
print(f'{len(labeled):,} of {len(raw):,} records belong to a labelled period')

33,819 of 49,605 records belong to a labelled period


In [ ]:
def keep_main_source_per_day(df):
    """iPhone and Watch can record the same steps: keep the source with most steps per day"""
    totals = df.groupby(['date', 'source'])['steps'].sum().reset_index()
    best = totals.loc[totals.groupby('date')['steps'].idxmax(), ['date', 'source']]
    return df.merge(best, on=['date', 'source'])

deduped = keep_main_source_per_day(labeled)

hourly = (deduped.pivot_table(index='date', columns='hour', values='steps', aggfunc='sum', fill_value=0)
          .reindex(columns=range(24), fill_value=0))
hourly.columns = [f'h{h:02d}' for h in hourly.columns]

meta = deduped.groupby('date')[['city', 'period']].first()
daily = meta.join(hourly)
daily = daily[daily[[c for c in daily.columns if c.startswith('h')]].sum(axis=1) >= MIN_STEPS_PER_DAY]
daily = daily.reset_index()
print(daily.groupby(['city', 'period']).size())

city    period
Almaty  ALM1      408
        ALM2      120
        ALM3       27
        ALM4       88
SF      SF1       231
        SF2        77
        SF3       121
        SF4        39
dtype: int64


In [16]:
# Sanity check: with correct local time, the busiest hours should look similar in both cities
HOUR_COLS = [f'h{h:02d}' for h in range(24)]
profile = daily.groupby('city')[HOUR_COLS].mean()
print('Busiest hour (local) per city:', profile.idxmax(axis=1).to_dict())
print('Quietest hour (local) per city:', profile.idxmin(axis=1).to_dict())
daily.to_csv(OUT_PATH, index=False)
print(f'saved {OUT_PATH}: {len(daily)} days')

Busiest hour (local) per city: {'Almaty': 'h12', 'SF': 'h13'}
Quietest hour (local) per city: {'Almaty': 'h03', 'SF': 'h04'}
saved data/daily_hourly.csv: 1111 days


In [17]:
print(deduped.groupby(['city', 'source'])['steps'].sum().unstack(0).round())
print(deduped.groupby(['city', 'source'])['date'].nunique().unstack(0))

city               Almaty         SF
source                              
Assyl’s iPhone  2592793.0  5428335.0
city            Almaty   SF
source                     
Assyl’s iPhone     710  475
